# Agents & the Runner Loop

This notebook accompanies the [Agent Foundry](https://agent-foundry-theta.vercel.app) OpenAI Agents SDK roadmap.

You will learn how to configure agents with `Agent(name, instructions)`, run them with `Runner.run_sync()` and `Runner.run()`, and inspect `RunResult`.

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Creating an Agent

An `Agent` is configured with a `name` and `instructions` (the system prompt). Optionally set `model` to pick which OpenAI model to use.

In [ ]:
from agents import Agent, Runner

agent = Agent(
    name="Research Assistant",
    instructions="""You are a research assistant. When asked about a topic:
1. Provide a clear, structured summary
2. Include key facts and figures
3. Keep your response concise""",
    model="gpt-4o",
)

print(f"Agent name: {agent.name}")
print(f"Model: {agent.model}")

## 4. Running Synchronously with Runner.run_sync()

`Runner.run_sync()` is the simplest way to run an agent. It blocks until the agent produces a final output.

In [ ]:
result = Runner.run_sync(agent, "What are the three laws of thermodynamics?")
print(result.final_output)

## 5. Running Asynchronously with Runner.run()

For production applications, use `await Runner.run()`. In Jupyter notebooks, you can `await` directly.

In [ ]:
result = await Runner.run(agent, "Explain quantum entanglement in simple terms.")
print(result.final_output)

## 6. Controlling the Loop with max_turns

Use `max_turns` to limit how many iterations the agent loop can take, preventing runaway loops.

In [ ]:
result = Runner.run_sync(
    agent,
    "Summarize the history of computing in one paragraph.",
    max_turns=5,
)
print(result.final_output)

## 7. Inspecting RunResult

The `RunResult` object contains `final_output`, `last_agent`, and `new_items` (the full trace of messages and tool calls).

In [ ]:
result = Runner.run_sync(agent, "What is Moore's Law?")

print(f"Final output: {result.final_output[:150]}...")
print(f"Last agent: {result.last_agent.name}")
print(f"Number of new items: {len(result.new_items)}")

print("\nAll items:")
for item in result.new_items:
    print(item)

## 8. Different Instructions, Different Behavior

The `instructions` shape how the agent responds. Compare two agents with different instructions on the same prompt.

In [ ]:
concise_agent = Agent(
    name="Concise Bot",
    instructions="Answer in exactly one sentence. Be direct and factual.",
)

detailed_agent = Agent(
    name="Detailed Bot",
    instructions="Provide thorough, well-structured answers with examples and context. Use bullet points.",
)

question = "What is machine learning?"

r1 = Runner.run_sync(concise_agent, question)
r2 = Runner.run_sync(detailed_agent, question)

print("=== Concise Bot ===")
print(r1.final_output)
print("\n=== Detailed Bot ===")
print(r2.final_output)

## Key Takeaways

- `Agent(name, instructions)` defines who the agent is and how it behaves
- `Runner.run_sync()` runs the agent synchronously; `Runner.run()` is the async version
- `RunResult` gives you `final_output`, `last_agent`, and `new_items` for full introspection
- `max_turns` limits loop iterations to prevent runaway agents
- Instructions are the primary lever for controlling agent behavior